
# Inactive Removal & Sampling

In polymer network modeling via the Maximum Entropy Homogenization Procedure (MEHP)
and Force Balance (:cite:t:`gusev_numerical_2019`, :cite:t:`bernhard_phantom_2025`),
networks generated below or near full conversion contain significant fractions of
**dangling chains**, **sol fractions**, and **topological loops**.

In :class:`~pylimer_tools_cpp.MEHPForceBalance2`, entanglements are modeled as static
topological constraints (fixed 4-functional links or Hookean springs) without contour
slipping. This introduces two interrelated and critical modeling questions:

1. **The Sampling Trap (Pre-Filter vs. Unfiltered Sampling)**:
   If entanglements are sampled without filtering (:meth:`~pylimer_tools_cpp.MCUniverseGenerator.sample_entanglements`
   with ``filtering_mode=EntanglementFilteringMode.NONE``), entanglements are placed indiscriminately
   along all chains, including dangling ends and unattached sol molecules.
   Because links cannot slip off open chain ends in Force Balance 2, an entanglement linking
   a dangling chain to the gel effectively pins the dangling arm in space, artificially
   transforming dead-end chains into load-bearing network strands!
   Near the percolation threshold ($p \approx 0.70 - 0.85$), this produces a massive,
   unphysical inflation of the predicted shear modulus ($G$ overestimated by 300% to > 4000%).

2. **Iterative Inactive Removal & Graph Simplification**:
   Can the force balance relaxation procedure itself detect and eliminate these unphysical constraints?
   Yes! :class:`~pylimer_tools_cpp.MEHPForceBalance2` provides dedicated structural simplification modes
   (:class:`~pylimer_tools_cpp.StructureSimplificationMode`):

   - ``NO_SIMPLIFICATION``:
     Retains all nodes, strands, and springs. Dangling ends relax their internal tension to zero,
     but any static entanglements pinning them remain active constraints.
   - ``INACTIVE_ONLY``:
     Iteratively detects inactive strands where end-to-end partial spring extensions are below
     a designated cutoff (``inactive_removal_cutoff``) or have dangling connectivity (degree 1),
     purging their springs and associated entanglement springs from the matrix.
   - ``INACTIVE_THEN_X2F``:
     First removes all inactive strands and entanglements, then systematically collapses 2-functional
     crosslinkers and intermediate nodes (unlinking bifunctional nodes). Two springs in series with
     lengths $N_1, N_2$ are replaced by a single equivalent spring of length $N_1 + N_2$.
     This compresses the sparse linear system to only the elastically active junctions ($f \ge 3$).

This example provides an in-depth statistical investigation using large polymer networks
($N_{\text{strands}} = 2{,}000$, $N_{\text{atoms}} \approx 100{,}000$, $N_{\text{ent}} \approx 7{,}500$)
across conversion $p \in [0.70, 0.95]$ with multi-seed ensembles and 95% confidence intervals:

- **Modulus Convergence**: How both topological pre-filtering and solver-level inactive removal recover
  the identical, physically correct equilibrium phantom modulus $G(p)$.
- **Topological Reduction**: How inactive pruning and 2-functional unlinking reduce the network size
  by up to 80%+ near percolation.
- **Degrees of Freedom & Solver Complexity**: The substantial memory and matrix dimension reductions achieved.
- **Practical Guidelines**: When to pre-filter, when to simplify, and recommended configurations.


## Multi-Seed Ensemble Simulation Procedure
We generate polymer networks across a conversion range from $p = 0.70$ (near the gelation threshold)
to $p = 0.95$ (dense, well-cured network).
For each conversion and Monte Carlo realization, we evaluate 5 distinct modeling regimes:

1. **Unfiltered + No Simplification** (``NO_SIMPLIFICATION``):
   Naive baseline where dangling chains receive entanglements and are retained in the matrix.
2. **Unfiltered + Inactive Pruning** (``INACTIVE_ONLY``):
   Entanglements sampled everywhere, but inactive strands and dead-end entanglements are pruned during relaxation.
3. **Unfiltered + Full Simplification** (``INACTIVE_THEN_X2F``):
   Inactive pruning followed by 2-functional node unlinking.
4. **Filtered + No Simplification** (``NO_SIMPLIFICATION``):
   Topological leaf peeling during sampling (no entanglements on dangling chains), unsimplified solver.
5. **Filtered + Full Simplification** (``INACTIVE_THEN_X2F``):
   Clean sampling combined with full topological simplification.

Results are cached in a committed ``.npz`` file for instantaneous CI and
documentation builds.



In [ ]:
import pylimer_tools_cpp as pt
import numpy as np
import matplotlib.ticker as ticker
import matplotlib.pyplot as plt
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path
import sys
from typing import Any, Dict, Optional, Sequence, Tuple

# Ensure examples/ is on sys.path when script is executed directly
if "__file__" in globals():
    _examples_dir = str(Path(__file__).resolve().parents[1])
    if _examples_dir not in sys.path:
        sys.path.insert(0, _examples_dir)


from _example_cache import (  # noqa: E402
    compute_ensemble_statistics,
    configure_process_threading,
    get_example_cache_path,
    load_or_compute_cache,
    resolve_max_workers,
    worker_process_initializer,
)
from pylimer_tools.io.bead_spring_parameter_provider import (  # noqa: E402
    ParameterType,
    get_parameters_for_polymer,
)


def _simulate_inactive_removal_single_repeat(
    r: int,
    seed: int,
    box_l: float,
    b: float,
    n_strands: int,
    strand_len: int,
    conversions: Sequence[float],
    n_ent: int,
    cutoff: float,
    r02: float,
    vol: float,
    gamma_conv: float,
    inactive_cutoff: float,
) -> Tuple[
    int,
    np.ndarray,
    np.ndarray,
    np.ndarray,
    np.ndarray,
    np.ndarray,
    np.ndarray,
    np.ndarray,
    np.ndarray,
    np.ndarray,
    np.ndarray,
    np.ndarray,
    np.ndarray,
    np.ndarray,
    np.ndarray,
    np.ndarray,
    np.ndarray,
    np.ndarray,
    np.ndarray,
    np.ndarray,
    np.ndarray,
]:
    """Execute a single Monte Carlo realization across conversions and simplification modes."""
    n_conv = len(conversions)

    # Modulus arrays [kPa]
    g_unf_nosimp = np.zeros(n_conv)
    g_unf_inact = np.zeros(n_conv)
    g_unf_x2f = np.zeros(n_conv)
    g_filt_nosimp = np.zeros(n_conv)
    g_filt_x2f = np.zeros(n_conv)

    # Spring count arrays
    sp_unf_nosimp = np.zeros(n_conv)
    sp_unf_inact = np.zeros(n_conv)
    sp_unf_x2f = np.zeros(n_conv)
    sp_filt_nosimp = np.zeros(n_conv)
    sp_filt_x2f = np.zeros(n_conv)

    # Link count arrays
    links_unf_nosimp = np.zeros(n_conv)
    links_unf_inact = np.zeros(n_conv)
    links_unf_x2f = np.zeros(n_conv)
    links_filt_nosimp = np.zeros(n_conv)
    links_filt_x2f = np.zeros(n_conv)

    # Iteration count arrays
    iter_unf_nosimp = np.zeros(n_conv)
    iter_unf_inact = np.zeros(n_conv)
    iter_unf_x2f = np.zeros(n_conv)
    iter_filt_nosimp = np.zeros(n_conv)
    iter_filt_x2f = np.zeros(n_conv)

    for i, conv in enumerate(conversions):
        gen = pt.MCUniverseGenerator(box_l, box_l, box_l)
        gen.set_seed(seed + i * 1000)
        gen.set_bead_distance(b)
        gen.set_mean_squared_bead_distance(b**2)
        gen.add_strands(n_strands, [strand_len] * n_strands)
        gen.add_crosslinkers(int(0.5 * n_strands), 4)
        gen.link_strands_to_conversion(conv)

        # Entanglement sampling: filtered vs. unfiltered
        es_filt = gen.sample_entanglements(
            n_ent,
            cutoff,
            0.0,
            0,
            3.0,
            f"s_filt_{seed}_{conv}",
            filtering_mode=pt.EntanglementFilteringMode.OMIT_INACTIVE,
        )
        es_unf = gen.sample_entanglements(
            n_ent,
            cutoff,
            0.0,
            0,
            3.0,
            f"s_unf_{seed}_{conv}",
            filtering_mode=pt.EntanglementFilteringMode.NONE,
        )

        # 1. Unfiltered + No Simplification
        fb1 = gen.get_force_balance2(es_unf)
        fb1.run_force_relaxation(
            simplification_mode=pt.StructureSimplificationMode.NO_SIMPLIFICATION
        )
        g_unf_nosimp[i] = (
            gamma_conv * np.sum(fb1.get_gamma_factors(r02)) / vol) * 1e6
        sp_unf_nosimp[i] = fb1.network.nr_of_springs
        links_unf_nosimp[i] = fb1.network.nr_of_links
        iter_unf_nosimp[i] = fb1.get_nr_of_iterations()

        # 2. Unfiltered + INACTIVE_ONLY
        fb2 = gen.get_force_balance2(es_unf)
        fb2.run_force_relaxation(
            simplification_mode=pt.StructureSimplificationMode.INACTIVE_ONLY,
            inactive_removal_cutoff=inactive_cutoff,
        )
        g_unf_inact[i] = (
            gamma_conv * np.sum(fb2.get_gamma_factors(r02)) / vol) * 1e6
        sp_unf_inact[i] = fb2.network.nr_of_springs
        links_unf_inact[i] = fb2.network.nr_of_links
        iter_unf_inact[i] = fb2.get_nr_of_iterations()

        # 3. Unfiltered + INACTIVE_THEN_X2F
        fb3 = gen.get_force_balance2(es_unf)
        fb3.run_force_relaxation(
            simplification_mode=pt.StructureSimplificationMode.INACTIVE_THEN_X2F,
            inactive_removal_cutoff=inactive_cutoff,
        )
        g_unf_x2f[i] = (
            gamma_conv * np.sum(fb3.get_gamma_factors(r02)) / vol) * 1e6
        sp_unf_x2f[i] = fb3.network.nr_of_springs
        links_unf_x2f[i] = fb3.network.nr_of_links
        iter_unf_x2f[i] = fb3.get_nr_of_iterations()

        # 4. Filtered + No Simplification
        fb4 = gen.get_force_balance2(es_filt)
        fb4.run_force_relaxation(
            simplification_mode=pt.StructureSimplificationMode.NO_SIMPLIFICATION
        )
        g_filt_nosimp[i] = (
            gamma_conv * np.sum(fb4.get_gamma_factors(r02)) / vol) * 1e6
        sp_filt_nosimp[i] = fb4.network.nr_of_springs
        links_filt_nosimp[i] = fb4.network.nr_of_links
        iter_filt_nosimp[i] = fb4.get_nr_of_iterations()

        # 5. Filtered + INACTIVE_THEN_X2F
        fb5 = gen.get_force_balance2(es_filt)
        fb5.run_force_relaxation(
            simplification_mode=pt.StructureSimplificationMode.INACTIVE_THEN_X2F,
            inactive_removal_cutoff=inactive_cutoff,
        )
        g_filt_x2f[i] = (
            gamma_conv * np.sum(fb5.get_gamma_factors(r02)) / vol) * 1e6
        sp_filt_x2f[i] = fb5.network.nr_of_springs
        links_filt_x2f[i] = fb5.network.nr_of_links
        iter_filt_x2f[i] = fb5.get_nr_of_iterations()

    return (
        r,
        g_unf_nosimp,
        g_unf_inact,
        g_unf_x2f,
        g_filt_nosimp,
        g_filt_x2f,
        sp_unf_nosimp,
        sp_unf_inact,
        sp_unf_x2f,
        sp_filt_nosimp,
        sp_filt_x2f,
        links_unf_nosimp,
        links_unf_inact,
        links_unf_x2f,
        links_filt_nosimp,
        links_filt_x2f,
        iter_unf_nosimp,
        iter_unf_inact,
        iter_unf_x2f,
        iter_filt_nosimp,
        iter_filt_x2f,
    )


def run_inactive_removal_and_sampling_simulation(
    conversions: Sequence[float] = (0.70, 0.75, 0.80, 0.85, 0.90, 0.95),
    n_strands: int = 2000,
    strand_len: int = 50,
    box_l: float = 50.0,
    n_repeats: int = 8,
    base_seed: int = 42,
    inactive_cutoff: float = 1e-4,
    max_workers: Optional[int] = None,
) -> Dict[str, Any]:
    """Execute multi-seed ensemble simulation comparing inactive strand removal and sampling."""
    params = get_parameters_for_polymer(
        "PDMS", parameter_type=ParameterType.GAUSSIAN)
    b = params.get("<b>").to(params.get("distance_units")).magnitude
    cutoff = params.get_sampling_cutoff()
    ent_density = params.get_entanglement_density()
    r02 = params.get("R02").to(params.get("distance_units") ** 2).magnitude
    gamma_conv = (
        (params.get("T") * params.get("kb") / (params.get("distance_units") ** 3))
        .to("MPa")
        .magnitude
    )

    vol = box_l**3
    n_ent = int(ent_density * vol)
    effective_workers = resolve_max_workers(max_workers)

    print(
        f"Simulating Inactive Removal & Sampling study: {n_repeats} repeats, "
        f"{len(conversions)} conversions, N_strands={n_strands}, N_ent={n_ent} across {effective_workers} workers..."
    )

    n_conv = len(conversions)
    all_g_unf_nosimp = np.zeros((n_conv, n_repeats))
    all_g_unf_inact = np.zeros((n_conv, n_repeats))
    all_g_unf_x2f = np.zeros((n_conv, n_repeats))
    all_g_filt_nosimp = np.zeros((n_conv, n_repeats))
    all_g_filt_x2f = np.zeros((n_conv, n_repeats))

    all_sp_unf_nosimp = np.zeros((n_conv, n_repeats))
    all_sp_unf_inact = np.zeros((n_conv, n_repeats))
    all_sp_unf_x2f = np.zeros((n_conv, n_repeats))
    all_sp_filt_nosimp = np.zeros((n_conv, n_repeats))
    all_sp_filt_x2f = np.zeros((n_conv, n_repeats))

    all_links_unf_nosimp = np.zeros((n_conv, n_repeats))
    all_links_unf_inact = np.zeros((n_conv, n_repeats))
    all_links_unf_x2f = np.zeros((n_conv, n_repeats))
    all_links_filt_nosimp = np.zeros((n_conv, n_repeats))
    all_links_filt_x2f = np.zeros((n_conv, n_repeats))

    all_iter_unf_nosimp = np.zeros((n_conv, n_repeats))
    all_iter_unf_inact = np.zeros((n_conv, n_repeats))
    all_iter_unf_x2f = np.zeros((n_conv, n_repeats))
    all_iter_filt_nosimp = np.zeros((n_conv, n_repeats))
    all_iter_filt_x2f = np.zeros((n_conv, n_repeats))

    tasks = [
        (
            r,
            base_seed + r * 100,
            box_l,
            b,
            n_strands,
            strand_len,
            conversions,
            n_ent,
            cutoff,
            r02,
            vol,
            gamma_conv,
            inactive_cutoff,
        )
        for r in range(n_repeats)
    ]

    configure_process_threading(effective_workers)

    if effective_workers > 1:
        with ProcessPoolExecutor(
            max_workers=effective_workers,
            initializer=worker_process_initializer,
        ) as executor:
            futures = [
                executor.submit(_simulate_inactive_removal_single_repeat, *t)
                for t in tasks
            ]
            completed = 0
            for fut in as_completed(futures):
                (
                    r,
                    g1,
                    g2,
                    g3,
                    g4,
                    g5,
                    sp1,
                    sp2,
                    sp3,
                    sp4,
                    sp5,
                    lk1,
                    lk2,
                    lk3,
                    lk4,
                    lk5,
                    it1,
                    it2,
                    it3,
                    it4,
                    it5,
                ) = fut.result()
                all_g_unf_nosimp[:, r] = g1
                all_g_unf_inact[:, r] = g2
                all_g_unf_x2f[:, r] = g3
                all_g_filt_nosimp[:, r] = g4
                all_g_filt_x2f[:, r] = g5

                all_sp_unf_nosimp[:, r] = sp1
                all_sp_unf_inact[:, r] = sp2
                all_sp_unf_x2f[:, r] = sp3
                all_sp_filt_nosimp[:, r] = sp4
                all_sp_filt_x2f[:, r] = sp5

                all_links_unf_nosimp[:, r] = lk1
                all_links_unf_inact[:, r] = lk2
                all_links_unf_x2f[:, r] = lk3
                all_links_filt_nosimp[:, r] = lk4
                all_links_filt_x2f[:, r] = lk5

                all_iter_unf_nosimp[:, r] = it1
                all_iter_unf_inact[:, r] = it2
                all_iter_unf_x2f[:, r] = it3
                all_iter_filt_nosimp[:, r] = it4
                all_iter_filt_x2f[:, r] = it5

                completed += 1
                print(
                    f"  [Inactive Removal] Completed repeat {completed}/{
                        n_repeats
                    } (repeat index {r + 1})..."
                )
    else:
        for t in tasks:
            (
                r,
                g1,
                g2,
                g3,
                g4,
                g5,
                sp1,
                sp2,
                sp3,
                sp4,
                sp5,
                lk1,
                lk2,
                lk3,
                lk4,
                lk5,
                it1,
                it2,
                it3,
                it4,
                it5,
            ) = _simulate_inactive_removal_single_repeat(*t)
            all_g_unf_nosimp[:, r] = g1
            all_g_unf_inact[:, r] = g2
            all_g_unf_x2f[:, r] = g3
            all_g_filt_nosimp[:, r] = g4
            all_g_filt_x2f[:, r] = g5

            all_sp_unf_nosimp[:, r] = sp1
            all_sp_unf_inact[:, r] = sp2
            all_sp_unf_x2f[:, r] = sp3
            all_sp_filt_nosimp[:, r] = sp4
            all_sp_filt_x2f[:, r] = sp5

            all_links_unf_nosimp[:, r] = lk1
            all_links_unf_inact[:, r] = lk2
            all_links_unf_x2f[:, r] = lk3
            all_links_filt_nosimp[:, r] = lk4
            all_links_filt_x2f[:, r] = lk5

            all_iter_unf_nosimp[:, r] = it1
            all_iter_unf_inact[:, r] = it2
            all_iter_unf_x2f[:, r] = it3
            all_iter_filt_nosimp[:, r] = it4
            all_iter_filt_x2f[:, r] = it5
            print(
                f"  [Inactive Removal] Completed repeat {
                    r + 1}/{n_repeats}...")

    # Compute ensemble statistics
    stat_g_unf_nosimp = compute_ensemble_statistics(all_g_unf_nosimp, axis=1)
    stat_g_unf_inact = compute_ensemble_statistics(all_g_unf_inact, axis=1)
    stat_g_unf_x2f = compute_ensemble_statistics(all_g_unf_x2f, axis=1)
    stat_g_filt_nosimp = compute_ensemble_statistics(all_g_filt_nosimp, axis=1)
    stat_g_filt_x2f = compute_ensemble_statistics(all_g_filt_x2f, axis=1)

    stat_sp_unf_nosimp = compute_ensemble_statistics(all_sp_unf_nosimp, axis=1)
    stat_sp_unf_inact = compute_ensemble_statistics(all_sp_unf_inact, axis=1)
    stat_sp_unf_x2f = compute_ensemble_statistics(all_sp_unf_x2f, axis=1)
    stat_sp_filt_nosimp = compute_ensemble_statistics(
        all_sp_filt_nosimp, axis=1)
    stat_sp_filt_x2f = compute_ensemble_statistics(all_sp_filt_x2f, axis=1)

    stat_links_unf_nosimp = compute_ensemble_statistics(
        all_links_unf_nosimp, axis=1)
    stat_links_unf_inact = compute_ensemble_statistics(
        all_links_unf_inact, axis=1)
    stat_links_unf_x2f = compute_ensemble_statistics(all_links_unf_x2f, axis=1)
    stat_links_filt_nosimp = compute_ensemble_statistics(
        all_links_filt_nosimp, axis=1)
    stat_links_filt_x2f = compute_ensemble_statistics(
        all_links_filt_x2f, axis=1)

    stat_iter_unf_nosimp = compute_ensemble_statistics(
        all_iter_unf_nosimp, axis=1)
    stat_iter_unf_inact = compute_ensemble_statistics(
        all_iter_unf_inact, axis=1)
    stat_iter_unf_x2f = compute_ensemble_statistics(all_iter_unf_x2f, axis=1)
    stat_iter_filt_nosimp = compute_ensemble_statistics(
        all_iter_filt_nosimp, axis=1)
    stat_iter_filt_x2f = compute_ensemble_statistics(all_iter_filt_x2f, axis=1)

    return {
        "conversions": np.array(conversions),
        "n_repeats": np.array(n_repeats),
        "n_strands": np.array(n_strands),
        "strand_len": np.array(strand_len),
        "box_l": np.array(box_l),
        "vol": np.array(vol),
        "n_ent": np.array(n_ent),
        # Modulus
        "g_unf_nosimp_mean": stat_g_unf_nosimp["mean"],
        "g_unf_nosimp_ci": stat_g_unf_nosimp["ci"],
        "g_unf_nosimp_all": all_g_unf_nosimp,
        "g_unf_inact_mean": stat_g_unf_inact["mean"],
        "g_unf_inact_ci": stat_g_unf_inact["ci"],
        "g_unf_inact_all": all_g_unf_inact,
        "g_unf_x2f_mean": stat_g_unf_x2f["mean"],
        "g_unf_x2f_ci": stat_g_unf_x2f["ci"],
        "g_unf_x2f_all": all_g_unf_x2f,
        "g_filt_nosimp_mean": stat_g_filt_nosimp["mean"],
        "g_filt_nosimp_ci": stat_g_filt_nosimp["ci"],
        "g_filt_nosimp_all": all_g_filt_nosimp,
        "g_filt_x2f_mean": stat_g_filt_x2f["mean"],
        "g_filt_x2f_ci": stat_g_filt_x2f["ci"],
        "g_filt_x2f_all": all_g_filt_x2f,
        # Springs
        "sp_unf_nosimp_mean": stat_sp_unf_nosimp["mean"],
        "sp_unf_inact_mean": stat_sp_unf_inact["mean"],
        "sp_unf_x2f_mean": stat_sp_unf_x2f["mean"],
        "sp_filt_nosimp_mean": stat_sp_filt_nosimp["mean"],
        "sp_filt_x2f_mean": stat_sp_filt_x2f["mean"],
        # Links
        "links_unf_nosimp_mean": stat_links_unf_nosimp["mean"],
        "links_unf_inact_mean": stat_links_unf_inact["mean"],
        "links_unf_x2f_mean": stat_links_unf_x2f["mean"],
        "links_filt_nosimp_mean": stat_links_filt_nosimp["mean"],
        "links_filt_x2f_mean": stat_links_filt_x2f["mean"],
        # Iterations
        "iter_unf_nosimp_mean": stat_iter_unf_nosimp["mean"],
        "iter_unf_inact_mean": stat_iter_unf_inact["mean"],
        "iter_unf_x2f_mean": stat_iter_unf_x2f["mean"],
        "iter_filt_nosimp_mean": stat_iter_filt_nosimp["mean"],
        "iter_filt_x2f_mean": stat_iter_filt_x2f["mean"],
    }

## Loading Cached Data or Recomputing
We load precomputed multi-seed ensemble results from the committed cache archive,
or execute the live parallel simulation when ``--recompute`` is requested.



In [ ]:
if __name__ == "__main__" or "sphinx_gallery" in sys.modules:
    cache_path = get_example_cache_path(
        "inactive_removal_and_sampling_cache.npz", "force_balance"
    )
    data = load_or_compute_cache(
        cache_path, run_inactive_removal_and_sampling_simulation
    )

    convs = data["conversions"]
    n_rep = int(data.get("n_repeats", 8))

    g_unf_nosimp_m = data["g_unf_nosimp_mean"]
    g_unf_nosimp_ci = data["g_unf_nosimp_ci"]
    g_unf_inact_m = data["g_unf_inact_mean"]
    g_unf_inact_ci = data["g_unf_inact_ci"]
    g_unf_x2f_m = data["g_unf_x2f_mean"]
    g_unf_x2f_ci = data["g_unf_x2f_ci"]
    g_filt_nosimp_m = data["g_filt_nosimp_mean"]
    g_filt_nosimp_ci = data["g_filt_nosimp_ci"]
    g_filt_x2f_m = data["g_filt_x2f_mean"]
    g_filt_x2f_ci = data["g_filt_x2f_ci"]

    sp_unf_nosimp_m = data["sp_unf_nosimp_mean"]
    sp_unf_inact_m = data["sp_unf_inact_mean"]
    sp_unf_x2f_m = data["sp_unf_x2f_mean"]
    sp_filt_nosimp_m = data["sp_filt_nosimp_mean"]
    sp_filt_x2f_m = data["sp_filt_x2f_mean"]

    links_unf_nosimp_m = data["links_unf_nosimp_mean"]
    links_unf_x2f_m = data["links_unf_x2f_mean"]
    links_filt_x2f_m = data["links_filt_x2f_mean"]

    iter_unf_nosimp_m = data["iter_unf_nosimp_mean"]
    iter_unf_inact_m = data["iter_unf_inact_mean"]
    iter_unf_x2f_m = data["iter_unf_x2f_mean"]
    iter_filt_nosimp_m = data["iter_filt_nosimp_mean"]
    iter_filt_x2f_m = data["iter_filt_x2f_mean"]

## Figure 1: Shear Modulus & The Dangling Chain Entanglement Trap
The left panel displays the equilibrium shear modulus $G$ across conversion.
Notice how the naive baseline (**Unfiltered Sampling + NO_SIMPLIFICATION**) severely
overestimates $G$ below $p \approx 0.90$ due to pinned dangling chains.

Once inactive strands are pruned or dangling chains are pre-filtered, all curves
collapse cleanly onto the true physical phantom percolation curve.

The right panel shows the **relative error (inflation factor)**:

\begin{align}\text{Error} = \frac{G_{\text{unsimplified}} - G_{\text{true}}}{G_{\text{true}}} \times 100\%\end{align}

demonstrating how unphysical constraints inflate stiffness by hundreds
to thousands of percent!



In [ ]:
if __name__ == "__main__" or "sphinx_gallery" in sys.modules:
    # Palette
    c_naive = "#d62728"  # Red: Unfiltered NO_SIMPLIFICATION
    c_filt_raw = "#1f77b4"  # Blue: Filtered NO_SIMPLIFICATION
    c_inact = "#ff7f0e"  # Orange: Unfiltered INACTIVE_ONLY
    c_truth = "#2ca02c"  # Green: Filtered INACTIVE_THEN_X2F
    c_unf_x2f = "#9467bd"  # Purple: Unfiltered INACTIVE_THEN_X2F

    # Figure 1: Modulus vs Conversion
    fig, ax1 = plt.subplots(figsize=(7, 5))
    ax1.plot(
        convs,
        g_unf_nosimp_m,
        "o--",
        color=c_naive,
        lw=2.2,
        ms=6,
        label="Unfiltered Sampling (NO_SIMPLIFICATION)",
    )
    ax1.fill_between(
        convs,
        g_unf_nosimp_m - g_unf_nosimp_ci,
        g_unf_nosimp_m + g_unf_nosimp_ci,
        color=c_naive,
        alpha=0.15,
    )

    ax1.plot(
        convs,
        g_filt_nosimp_m,
        "s:",
        color=c_filt_raw,
        lw=2.0,
        ms=6,
        label="Filtered Sampling (NO_SIMPLIFICATION)",
    )
    ax1.fill_between(
        convs,
        g_filt_nosimp_m - g_filt_nosimp_ci,
        g_filt_nosimp_m + g_filt_nosimp_ci,
        color=c_filt_raw,
        alpha=0.15,
    )

    ax1.plot(
        convs,
        g_unf_inact_m,
        "^--",
        color=c_inact,
        lw=1.8,
        ms=6,
        label="Unfiltered Sampling (INACTIVE_ONLY)",
    )
    ax1.fill_between(
        convs,
        g_unf_inact_m - g_unf_inact_ci,
        g_unf_inact_m + g_unf_inact_ci,
        color=c_inact,
        alpha=0.15,
    )

    ax1.plot(
        convs,
        g_filt_x2f_m,
        "d-",
        color=c_truth,
        lw=2.5,
        ms=7,
        label="Filtered Sampling (INACTIVE_THEN_X2F)",
    )
    ax1.fill_between(
        convs,
        g_filt_x2f_m - g_filt_x2f_ci,
        g_filt_x2f_m + g_filt_x2f_ci,
        color=c_truth,
        alpha=0.2,
    )

    ax1.plot(
        convs,
        g_unf_x2f_m,
        "x-.",
        color=c_unf_x2f,
        lw=1.8,
        ms=6,
        label="Unfiltered Sampling (INACTIVE_THEN_X2F)",
    )

    ax1.set_xlabel("Crosslinker Conversion $p$", fontsize=12)
    ax1.set_ylabel("Equilibrium Shear Modulus $G$ [kPa]", fontsize=12)
    ax1.set_title(
        "Shear Modulus across Network Conversion", fontsize=13, fontweight="bold"
    )
    ax1.grid(True, linestyle=":", alpha=0.6)
    ax1.legend(loc="upper left", frameon=True, fontsize=9.5)
    plt.tight_layout()
    plt.show()

    # Figure 2: Artificial Inflation Factor (%)
    fig, ax2 = plt.subplots(figsize=(7, 5))
    g_truth = g_filt_x2f_m
    # Avoid division by near-zero at lowest conversion if unpercolated
    inflation_unf = np.where(
        g_truth > 0.1, (g_unf_nosimp_m - g_truth) / g_truth * 100.0, 0.0
    )
    inflation_filt_raw = np.where(
        g_truth > 0.1, (g_filt_nosimp_m - g_truth) / g_truth * 100.0, 0.0
    )

    ax2.plot(
        convs,
        inflation_unf,
        "o--",
        color=c_naive,
        lw=2.2,
        ms=6,
        label="Unfiltered (NO_SIMPLIFICATION) Error",
    )
    ax2.plot(
        convs,
        inflation_filt_raw,
        "s:",
        color=c_filt_raw,
        lw=2.0,
        ms=6,
        label="Filtered (NO_SIMPLIFICATION) Discrepancy",
    )
    ax2.axhline(0, color="black", linestyle="-", lw=1.0, alpha=0.7)

    ax2.set_xlabel("Crosslinker Conversion $p$", fontsize=12)
    ax2.set_ylabel(
        "Artificial Modulus Inflation $\\Delta G / G_{\\text{true}}$ [%]",
        fontsize=12,
    )
    ax2.set_title(
        "Artificial Stiffening from Trapped Dangling Chains",
        fontsize=13,
        fontweight="bold",
    )
    ax2.grid(True, linestyle=":", alpha=0.6)
    ax2.legend(loc="upper right", frameon=True, fontsize=10)

    plt.tight_layout()
    plt.show()

## Figure 2: Network Topology & Simplification Efficiency
How many springs and links actually carry elastic load, and how many are redundant dead ends?

- **Left Panel**: Total spring count $N_{\text{springs}}$ before and after simplification.
  Notice that below $p \approx 0.80$, `INACTIVE_THEN_X2F` eliminates up to 80% of all
  springs in the network!
- **Right Panel**: Pruned spring fraction ($1 - N_{\text{simplified}} / N_{\text{initial}}$).
  Even at high conversion ($p = 0.95$), 15–20% of springs are pruned or collapsed,
  reducing memory and solver overhead for all subsequent deformation steps.



In [ ]:
if __name__ == "__main__" or "sphinx_gallery" in sys.modules:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

    # Panel 1: Number of springs vs. conversion
    ax1.plot(
        convs,
        sp_unf_nosimp_m,
        "o--",
        color=c_naive,
        lw=2.0,
        ms=6,
        label="Initial Unfiltered Springs",
    )
    ax1.plot(
        convs,
        sp_filt_nosimp_m,
        "s:",
        color=c_filt_raw,
        lw=2.0,
        ms=6,
        label="Initial Filtered Springs",
    )
    ax1.plot(
        convs,
        sp_unf_inact_m,
        "^--",
        color=c_inact,
        lw=2.0,
        ms=6,
        label="Unfiltered + INACTIVE_ONLY",
    )
    ax1.plot(
        convs,
        sp_filt_x2f_m,
        "d-",
        color=c_truth,
        lw=2.5,
        ms=7,
        label="Filtered + INACTIVE_THEN_X2F",
    )
    ax1.plot(
        convs,
        sp_unf_x2f_m,
        "x-.",
        color=c_unf_x2f,
        lw=2.0,
        ms=6,
        label="Unfiltered + INACTIVE_THEN_X2F",
    )

    ax1.set_xlabel("Crosslinker Conversion $p$", fontsize=12)
    ax1.set_ylabel("Number of Springs $N_{\\text{springs}}$", fontsize=12)
    ax1.set_title(
        "Remaining Springs after Topological Pruning",
        fontsize=13,
        fontweight="bold",
    )
    ax1.grid(True, linestyle=":", alpha=0.6)
    ax1.legend(loc="center left", frameon=True, fontsize=10)

    # Panel 2: Pruned percentage
    pruned_unf_x2f = (1.0 - sp_unf_x2f_m / sp_unf_nosimp_m) * 100.0
    pruned_filt_x2f = (1.0 - sp_filt_x2f_m / sp_filt_nosimp_m) * 100.0

    ax2.plot(
        convs,
        pruned_unf_x2f,
        "x-.",
        color=c_unf_x2f,
        lw=2.2,
        ms=7,
        label="Unfiltered Pruning (INACTIVE_THEN_X2F)",
    )
    ax2.plot(
        convs,
        pruned_filt_x2f,
        "d-",
        color=c_truth,
        lw=2.2,
        ms=7,
        label="Filtered Pruning (INACTIVE_THEN_X2F)",
    )

    ax2.set_xlabel("Crosslinker Conversion $p$", fontsize=12)
    ax2.set_ylabel("Springs Pruned / Collapsed [%]", fontsize=12)
    ax2.set_title(
        "Network Simplification Fraction vs. Conversion",
        fontsize=13,
        fontweight="bold",
    )
    ax2.yaxis.set_major_formatter(ticker.PercentFormatter())
    ax2.grid(True, linestyle=":", alpha=0.6)
    ax2.legend(loc="upper right", frameon=True, fontsize=10.5)

    plt.tight_layout()
    plt.show()

## Figure 3: Linear System Dimension & Solver Relaxation Iterations
Simplification directly reduces the dimension of the linear system solved during force balance:

\begin{align}\mathbf{K} \mathbf{u} = \mathbf{f}, \quad \text{dim}(\mathbf{u}) = 3 N_{\text{links}}\end{align}

- **Left Panel**: Degrees of freedom ($3 N_{\text{links}}$) in the sparse linear system.
  Unlinking bifunctional nodes ($f = 2$) reduces the matrix dimension substantially.
- **Right Panel**: Number of BiCGSTAB solver iterations required for force relaxation.
  While unsimplified networks solve a fixed, slack system in ~500 iterations, the simplified
modes perform iterative pruning cycles, ensuring a strictly clean
topological structure.



In [ ]:
if __name__ == "__main__" or "sphinx_gallery" in sys.modules:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

    # Panel 1: Degrees of freedom (3 * N_links)
    ax1.plot(
        convs,
        3 * links_unf_nosimp_m,
        "o--",
        color=c_naive,
        lw=2.0,
        ms=6,
        label="Unfiltered Unsimplified (3 $N_{\\text{links}}$)",
    )
    ax1.plot(
        convs,
        3 * links_filt_x2f_m,
        "d-",
        color=c_truth,
        lw=2.2,
        ms=7,
        label="Filtered + INACTIVE_THEN_X2F",
    )
    ax1.plot(
        convs,
        3 * links_unf_x2f_m,
        "x-.",
        color=c_unf_x2f,
        lw=2.0,
        ms=6,
        label="Unfiltered + INACTIVE_THEN_X2F",
    )

    ax1.set_xlabel("Crosslinker Conversion $p$", fontsize=12)
    ax1.set_ylabel(
        "Linear System Dimension ($3 N_{\\text{links}}$)",
        fontsize=12)
    ax1.set_title(
        "Degrees of Freedom in Sparse Force Balance",
        fontsize=13,
        fontweight="bold",
    )
    ax1.grid(True, linestyle=":", alpha=0.6)
    ax1.legend(loc="upper left", frameon=True, fontsize=10)

    # Panel 2: Solver Iterations
    ax2.plot(
        convs,
        iter_unf_nosimp_m,
        "o--",
        color=c_naive,
        lw=2.0,
        ms=6,
        label="Unfiltered NO_SIMPLIFICATION",
    )
    ax2.plot(
        convs,
        iter_filt_nosimp_m,
        "s:",
        color=c_filt_raw,
        lw=2.0,
        ms=6,
        label="Filtered NO_SIMPLIFICATION",
    )
    ax2.plot(
        convs,
        iter_unf_inact_m,
        "^--",
        color=c_inact,
        lw=2.0,
        ms=6,
        label="Unfiltered INACTIVE_ONLY",
    )
    ax2.plot(
        convs,
        iter_filt_x2f_m,
        "d-",
        color=c_truth,
        lw=2.2,
        ms=7,
        label="Filtered INACTIVE_THEN_X2F",
    )
    ax2.plot(
        convs,
        iter_unf_x2f_m,
        "x-.",
        color=c_unf_x2f,
        lw=2.0,
        ms=6,
        label="Unfiltered INACTIVE_THEN_X2F",
    )

    ax2.set_xlabel("Crosslinker Conversion $p$", fontsize=12)
    ax2.set_ylabel("Relaxation Iterations", fontsize=12)
    ax2.set_title(
        "Solver Convergence & Pruning Iterations",
        fontsize=13,
        fontweight="bold",
    )
    ax2.grid(True, linestyle=":", alpha=0.6)
    ax2.legend(loc="upper right", frameon=True, fontsize=9.5)

    plt.tight_layout()
    plt.show()

## Physical Interpretation & Practical Recommendations

1. **Why Unfiltered Entanglement Sampling Fails in Force Balance 2**:
   In classical slip-link simulations (:class:`~pylimer_tools_cpp.MEHPForceBalance` with slipping enabled),
   a slip-link placed on a dangling chain can dynamically slide off the free chain end during
   deformation. In contrast, :class:`~pylimer_tools_cpp.MEHPForceBalance2` models entanglements as
   static junctions. Without topological filtering, an entanglement between a dangling end and
   the gel artificially anchors the dangling chain, preventing it from relaxing and converting
   zero-modulus sol/dangling material into load-bearing network strands.
   At $p \le 0.85$, this artifact inflates $G$ by hundreds to thousands of percent!

2. **Topological Leaf Peeling is the First Line of Defense**:
   Passing ``filtering_mode=EntanglementFilteringMode.OMIT_INACTIVE`` (the default in
   :meth:`~pylimer_tools_cpp.MCUniverseGenerator.sample_entanglements` and
   :meth:`~pylimer_tools_cpp.MCUniverseGenerator.get_force_balance2`, or ``PRUNE_UPFRONT`` for active-only sampling)
   uses graph-theoretic leaf peeling to eliminate dangling and sol strands before entanglement pairs are sampled.
   This completely prevents artificial dangling entanglements.

3. **The Solver Pruning Rescues Corrupted Networks**:
   Even if entanglements were sampled on an unfiltered network, running :class:`~pylimer_tools_cpp.MEHPForceBalance2`
   with ``simplification_mode=StructureSimplificationMode.INACTIVE_THEN_X2F`` automatically detects
   all zero-tension dangling arms and prunes them, recovering the identical physical modulus
   ($G_{\text{unfiltered+X2F}} \equiv G_{\text{filtered+X2F}}$).

4. **Recommended Best Practice**:
   - Always keep filtering enabled (``filtering_mode=EntanglementFilteringMode.OMIT_INACTIVE`` or ``PRUNE_UPFRONT``) during entanglement sampling.
   - For subsequent large-scale deformation studies (e.g. uniaxial, biaxial, or shear strain sweeps),
     perform an initial relaxation with ``simplification_mode=StructureSimplificationMode.INACTIVE_THEN_X2F``.
     This permanently purges inactive degrees of freedom and unlinks bifunctional nodes, accelerating
     every successive deformation step by 20–80%.

